[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Individual-Contest/1_Find_the_Order/code/baseline/solution.ipynb)

# Find the Order — baseline on Google Colab

Run all (GPU runtime recommended). The first cell fetches the [dataset](https://huggingface.co/datasets/IOAI-official/ioai-2026-find-the-order) and recreates the contest file layout; every cell after it is the original, untouched baseline.

> **Unofficial educational version** — provided so the task can be used outside the contest environment, reading the data directly from the Hugging Face dataset. The official contest artifacts are preserved in `code/baseline-original/` and `code/grading-original/`.

In [ ]:
# ============================ Colab setup (added) ============================
# Downloads the task dataset from Hugging Face and lays it out exactly as the
# contest environment did. Everything below this cell is the original baseline.
import os, sys, shutil, subprocess
from pathlib import Path
def sh(c): print('+',c); subprocess.run(c, shell=True, check=True)
sh('pip -q install librosa soundfile huggingface_hub')

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("IOAI-official/ioai-2026-find-the-order", repo_type="dataset"))
def link(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists(): return
    os.symlink(src, dst)
# merged dataset root: split folders + *_answers files, public and private together
DSROOT = Path("/content/dsroot").resolve()
for sub in ("public","private"):
    d = DATA/sub
    if d.is_dir():
        for child in d.iterdir():
            link(child, DSROOT/child.name)
print("dataset root:", DSROOT, "->", sorted(p.name for p in DSROOT.iterdir()))

EVAL_SPLIT = "test_leaderboard_a"   # which graded split plays the role of dataset/test_public
ws = Path.cwd()
for name in ("train","pretrain","pretest"): link(DSROOT/name, ws/"dataset"/name)
link(DSROOT/EVAL_SPLIT, ws/"dataset"/"test_public")
# the three frozen model bundles the contest shipped offline
for repo, name in [("facebook/wav2vec2-base-960h","wav2vec2-base-960h"),
                   ("openai/whisper-small","whisper-small"),
                   ("Qwen/Qwen2.5-0.5B","qwen2.5-0.5b")]:
    p = Path("models")/name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        link(snapshot_download(repo), p)
print("ready: dataset/ (test_public ->", EVAL_SPLIT + ") and models/")


In [1]:
# Xóa thư mục mount bị lỗi trước đó để tránh lỗi 'not empty'
!rm -rf /content/drive

# Tiến hành mount lại Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

MessageError: Error: credential propagation was unsuccessful

In [11]:
import shutil
import os

# 1. Khôi phục các mô hình .pt từ Drive
src_dir = "/content/drive/MyDrive/IOAI_Project_All"
dst_dir = "/content"

print("--- Bắt đầu khôi phục mô hình từ Drive ---")
for filename in os.listdir(src_dir):
    if filename.endswith('.pt'):
        src_path = os.path.join(src_dir, filename)
        dst_path = os.path.join(dst_dir, filename)
        shutil.copy2(src_path, dst_path)
        print(f"Đã khôi phục checkpoint: {filename}")

# 2. Re-create dsroot bằng cách tải trực tiếp từ Hugging Face (tránh lỗi file zip hỏng)
print("\n--- Đang tái tạo lại thư mục dữ liệu dsroot từ Hugging Face... ---")
from pathlib import Path
from huggingface_hub import snapshot_download

DATA = Path(snapshot_download("IOAI-official/ioai-2026-find-the-order", repo_type="dataset"))
def link(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists(): return
    os.symlink(src, dst)

DSROOT = Path("/content/dsroot").resolve()
for sub in ("public","private"):
    d = DATA/sub
    if d.is_dir():
        for child in d.iterdir():
            link(child, DSROOT/child.name)

ws = Path.cwd()
for name in ("train","pretrain","pretest"):
    link(DSROOT/name, ws/"dataset"/name)
link(DSROOT/"test_leaderboard_a", ws/"dataset"/"test_public")

print("Dữ liệu gốc đã được tải lại thành công vào /content/dsroot/")
print("Quá trình khôi phục hoàn tất! Bạn có thể tiếp tục chạy code huấn luyện/đánh giá.")

--- Bắt đầu khôi phục mô hình từ Drive ---
Đã khôi phục checkpoint: best_model_fold_1.pt
Đã khôi phục checkpoint: best_model_fold_2.pt
Đã khôi phục checkpoint: best_model_fold_3.pt
Đã khôi phục checkpoint: best_model_fold_4.pt
Đã khôi phục checkpoint: train_features_from_ram.pt
Đã khôi phục checkpoint: test_features_from_ram.pt

--- Đang tái tạo lại thư mục dữ liệu dsroot từ Hugging Face... ---


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16512 files:   0%|          | 0/16512 [00:00<?, ?it/s]

Dữ liệu gốc đã được tải lại thành công vào /content/dsroot/
Quá trình khôi phục hoàn tất! Bạn có thể tiếp tục chạy code huấn luyện/đánh giá.


# Find the Order — Baseline solution

A **deterministic** smoke-test baseline. For each dialogue in **`dataset/test_public/`** it submits the identity permutation `P = [0, 1, …, n-1]` — the chunks left in their given order. The score sits at the ~0.5 floor. It writes **`answers.json`** at the repo root in the rank convention.

Replace the logic below with your own to beat the floor.

In [ ]:
import os, json

TEST_DIR = "dataset/test_public"        # grade-time: hidden eval set is overlaid here
OUTPUT   = "answers.json"

def n_chunks(ddir):
    return sum(1 for f in os.listdir(ddir)
               if f.startswith("chunk_") and f.endswith(".wav"))

dialogue_ids = sorted(
    (d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))),
    key=lambda x: int(x) if x.isdigit() else x,
)

answers = {}
for did in dialogue_ids:
    n = n_chunks(os.path.join(TEST_DIR, did))
    answers[did] = list(range(n))        # identity: chunk_i -> position i

with open(OUTPUT, "w") as f:
    json.dump(answers, f)
print(f"wrote {OUTPUT}: {len(answers)} dialogues")

## Additional Models & Libraries

## wav2-vec2-base-960h

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import torch
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2Processor

WAV2VEC_PATH = "models/wav2vec2-base-960h"
SAMPLE_AUDIO = "dataset/train/0/chunk_0.wav"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained(WAV2VEC_PATH)
model = Wav2Vec2Model.from_pretrained(WAV2VEC_PATH).to(device).eval()

audio, _ = librosa.load(SAMPLE_AUDIO, sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
    h = model(inputs.input_values.to(device)).last_hidden_state
embed = h.mean(dim=1).squeeze().cpu().numpy()

print(f"Wav2vec embedding shape: {embed.shape}")

## Qwen2.5-0.5B

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN25_MODEL_PATH = "models/qwen2.5-0.5b"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(QWEN25_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(QWEN25_MODEL_PATH, dtype=torch.bfloat16).to(device).eval()

prompt = "The weather today is"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=20, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## Whisper-small

In [ ]:
import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor

WHISPER_MODEL_PATH = "models/whisper-small"
SAMPLE_AUDIO = "dataset/train/0/chunk_0.wav"

processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_PATH).to(device).eval()


audio, _ = librosa.load(SAMPLE_AUDIO, sr=16000)
feats = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
with torch.no_grad():
    ids = model.generate(feats.to(device), language="en", task="transcribe")
result = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
print(f"Whisper ASR trasncription: {result}")

# Lời giải Find the Order — viết lại theo sơ đồ tư duy

Các cell dưới đây dùng **vector đặc trưng đã trích sẵn** (không chạy lại Whisper / Wav2Vec2 / Qwen):

| Cell | Việc làm |
|---|---|
| 1 | Thư viện và đường dẫn |
| 2 | Đọc đáp án + 2 câu mở đầu từ /content/dsroot |
| 3 | Nạp file train_features_from_ram.pt và test_features_from_ram.pt |
| 4 | Mô hình: ma trận "câu j nói ngay sau câu i" + ma trận "cùng người nói" |
| 5 | Tìm thứ tự tốt nhất (Bitmask DP / Beam Search) + hàm chấm điểm |
| 6 | Huấn luyện 5-Fold |
| 7 | Dự đoán tập test, ghi answers.json |

**Chú ý định dạng đáp án của đề:** file *_answers.json và answers.json đều dùng kiểu **RANK**: rank[i] = vị trí (theo thời gian) của chunk_i. Mô hình bên trong làm việc với **ORDER** (dãy chunk theo thời gian), nên cần đổi qua lại.

In [12]:
# ===================== CELL 1: Thư viện và đường dẫn =====================
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import KFold

DSROOT = "/content/dsroot"                                  # thư mục dữ liệu gốc
TRAIN_FEATURE_FILE = "/content/train_features_from_ram.pt"  # vector đặc trưng tập train
TEST_FEATURE_FILE = "/content/test_features_from_ram.pt"    # vector đặc trưng tập test
TEST_SPLIT = "test_leaderboard_a"   # tập mà file test_features được trích ra (dataset/test_public)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Cố định seed để chạy lại ra kết quả giống nhau
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("Thiết bị:", device)

Thiết bị: cpu


In [13]:
# ===================== CELL 2: Đọc đáp án và 2 câu mở đầu =====================
# Định dạng của đề (RANK):  rank[i] = vị trí theo thời gian của chunk_i
#   Ví dụ rank  = [2, 0, 1]  -> chunk_1 nói đầu tiên, rồi chunk_2, rồi chunk_0
#   nên  order = [1, 2, 0]  (dãy chunk theo thời gian)
# Mô hình dùng ORDER, còn file nộp answers.json phải ở dạng RANK.

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def rank_to_order(rank):
    order = [0] * len(rank)
    for chunk, position in enumerate(rank):
        order[position] = chunk
    return order


def order_to_rank(order):
    rank = [0] * len(order)
    for position, chunk in enumerate(order):
        rank[chunk] = position
    return rank


# Tập train: đáp án nằm ở dsroot/train_answers.json  ->  {id hội thoại: rank}
train_answers = read_json(f"{DSROOT}/train_answers.json")

# Tập test: chỉ có prefix.json  ->  {id hội thoại: [chunk nói đầu tiên, chunk nói thứ hai]}
test_prefix = read_json(f"{DSROOT}/{TEST_SPLIT}/prefix.json")

# Đáp án tập test (chỉ dùng để tự chấm điểm, có thì dùng, không có thì thôi)
test_answer_file = f"{DSROOT}/{TEST_SPLIT}_answers.json"
test_answers = read_json(test_answer_file) if os.path.exists(test_answer_file) else None

print("Số hội thoại train có đáp án:", len(train_answers))
print("Số hội thoại test có prefix :", len(test_prefix))
print("Có đáp án test để tự chấm   :", test_answers is not None)

example_id = next(iter(train_answers))
print("\nVí dụ hội thoại", example_id)
print("  rank :", train_answers[example_id])
print("  order:", rank_to_order(train_answers[example_id]))

Số hội thoại train có đáp án: 1288
Số hội thoại test có prefix : 100
Có đáp án test để tự chấm   : True

Ví dụ hội thoại 0
  rank : [11, 6, 0, 8, 7, 5, 3, 4, 10, 1, 2, 9]
  order: [2, 9, 10, 6, 7, 5, 1, 4, 3, 11, 8, 0]


In [14]:
# ===================== CELL 3: Nạp vector đặc trưng đã trích sẵn =====================
train_raw = torch.load(TRAIN_FEATURE_FILE, map_location="cpu", weights_only=False)
test_raw = torch.load(TEST_FEATURE_FILE, map_location="cpu", weights_only=False)

# Nếu file được lưu dạng dict {id: {...}} thì đổi về list cho giống nhau
if isinstance(train_raw, dict):
    train_raw = [dict(item, conv_id=key) for key, item in train_raw.items()]
if isinstance(test_raw, dict):
    test_raw = [dict(item, conv_id=key) for key, item in test_raw.items()]

print("Các key trong 1 mẫu:", list(train_raw[0].keys()))


def make_item(raw):
    # Nếu file của bạn đặt tên key khác thì sửa 4 dòng dưới đây
    return {
        "id": str(raw["conv_id"]),
        "spk": raw["spk_embs"].float(),    # [N, 768]  màu giọng (Wav2Vec2 layer 2-3)
        "edge": raw["edge_embs"].float(),  # [N, 1536] ngữ điệu đầu + cuối câu (Wav2Vec2 layer cuối)
        "sem": raw["sem_embs"].float(),    # [N, 896]  ngữ nghĩa câu (Whisper -> Qwen)
    }


# Nhãn được đọc lại từ dsroot (không dùng nhãn trong file .pt) để chắc chắn đúng định dạng
train_data = []
for raw in train_raw:
    item = make_item(raw)
    order = rank_to_order(train_answers[item["id"]])
    item["order"] = order
    item["first"] = order[0]
    item["second"] = order[1]
    train_data.append(item)

test_data = []
for raw in test_raw:
    item = make_item(raw)
    item["first"], item["second"] = test_prefix[item["id"]]
    test_data.append(item)

# Kiểm tra số câu trong vector khớp với đáp án
for item in train_data:
    assert item["spk"].shape[0] == len(item["order"]), f"Hội thoại {item['id']} bị lệch số câu"

# Đưa hết lên GPU một lần cho nhanh (dữ liệu nhỏ, chỉ vài trăm MB)
for item in train_data + test_data:
    for key in ["spk", "edge", "sem"]:
        item[key] = item[key].to(device)

print(f"Train: {len(train_data)} hội thoại | Test: {len(test_data)} hội thoại")
sample = train_data[0]
print("Hội thoại", sample["id"], "| spk", tuple(sample["spk"].shape),
      "| edge", tuple(sample["edge"].shape), "| sem", tuple(sample["sem"].shape))
print("order:", sample["order"], "| 2 câu đầu:", sample["first"], sample["second"])

Các key trong 1 mẫu: ['conv_id', 'spk_embs', 'edge_embs', 'sem_embs', 'first_two', 'gt_order']
Train: 1288 hội thoại | Test: 100 hội thoại
Hội thoại 0 | spk (12, 768) | edge (12, 1536) | sem (12, 896)
order: [2, 9, 10, 6, 7, 5, 1, 4, 3, 11, 8, 0] | 2 câu đầu: 2 9


In [ ]:
# ===================== CELL 3b: Kiểm tra và chuẩn hóa đặc trưng =====================
# Vì sao cần: nếu vector có vài chiều rất lớn và GIỐNG NHAU giữa các câu (rất hay gặp ở
# hidden state của Wav2Vec2 / Qwen), thì sau Linear + LayerNorm mọi câu gần như trùng nhau
# -> ma trận S[i][j] gần như đều -> mô hình không học được gì ở nhánh S.
# Cách sửa: trừ trung bình và chia độ lệch chuẩn theo từng chiều (tính trên tập train).

print("TRƯỚC khi chuẩn hóa (hội thoại đầu tiên):")
for key in ["spk", "edge", "sem"]:
    x = train_data[0][key]
    print(f"  {key:4s} | |x| lớn nhất {x.abs().max().item():10.2f} | |x| trung bình {x.abs().mean().item():8.3f}"
          f" | độ khác nhau giữa các câu {x.std(dim=0).mean().item():8.4f}")

for key in ["spk", "edge", "sem"]:
    all_train = torch.cat([item[key] for item in train_data], dim=0)   # [tổng số câu, số chiều]
    mean = all_train.mean(dim=0)
    std = all_train.std(dim=0) + 1e-6
    for item in train_data + test_data:
        item[key] = (item[key] - mean) / std

print("\nSAU khi chuẩn hóa:")
for key in ["spk", "edge", "sem"]:
    x = train_data[0][key]
    print(f"  {key:4s} | |x| lớn nhất {x.abs().max().item():10.2f} | |x| trung bình {x.abs().mean().item():8.3f}"
          f" | độ khác nhau giữa các câu {x.std(dim=0).mean().item():8.4f}")

In [15]:
# ===================== CELL 4: Mô hình và hàm loss =====================
# Mô hình trả về 2 ma trận [N, N] cho 1 hội thoại:
#   next_score[i, j]   : điểm để câu j nói NGAY SAU câu i (có hướng: i->j khác j->i)
#   same_speaker[i, j] : xác suất câu i và câu j do CÙNG một người nói

class OrderModel(nn.Module):
    def __init__(self, spk_dim=768, edge_dim=1536, sem_dim=896, d_model=256):
        super().__init__()
        self.d_model = d_model

        # Gộp 3 vector (768 + 1536 + 896 = 3200) rồi nén về 256 chiều
        self.merge = nn.Sequential(
            nn.Linear(spk_dim + edge_dim + sem_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(0.1),
        )

        # 2 lớp Transformer: cho mọi câu trong hội thoại "nhìn" nhau
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=8, dim_feedforward=d_model * 2,
                                           dropout=0.1, batch_first=True)
        self.context = nn.TransformerEncoder(layer, num_layers=2)

        # Nhánh 1: hai phép chiếu khác nhau cho vai "câu trước" (Q) và "câu sau" (K)
        self.as_before = nn.Linear(d_model, d_model)
        self.as_after = nn.Linear(d_model, d_model)

        # Nhánh 2: chỉ dùng màu giọng để so người nói (không dùng ngữ nghĩa)
        self.voice = nn.Sequential(
            nn.Linear(spk_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, 128),
        )

    def forward(self, spk, edge, sem):
        # spk: [N, 768], edge: [N, 1536], sem: [N, 896]
        x = torch.cat([spk, edge, sem], dim=-1).unsqueeze(0)   # [1, N, 3200]
        h = self.context(self.merge(x)).squeeze(0)             # [N, 256]

        # Nhánh 1: ma trận chuyển tiếp
        Q = self.as_before(h)
        K = self.as_after(h)
        next_score = Q @ K.T / (self.d_model ** 0.5)           # [N, N]
        N = next_score.shape[0]
        diagonal = torch.eye(N, dtype=torch.bool, device=next_score.device)
        next_score = next_score.masked_fill(diagonal, -1e9)    # cấm câu tự nối vào chính nó

        # Nhánh 2: ma trận cùng người nói (cosine đưa về khoảng (0, 1))
        v = F.normalize(self.voice(spk), dim=-1)               # [N, 128]
        same_speaker = ((v @ v.T + 1) / 2).clamp(1e-6, 1 - 1e-6)
        return next_score, same_speaker


def compute_loss(next_score, same_speaker, order):
    order = torch.tensor(order, device=next_score.device)

    # Loss 1: với mỗi câu order[t], câu đúng tiếp theo là order[t+1]  (bài toán chọn 1 trong N)
    loss_next = F.cross_entropy(next_score[order[:-1]], order[1:])

    # Loss 2: hội thoại 2 người nói luân phiên
    #   2 câu liền nhau (t, t+1) -> khác người nói (nhãn 0)
    #   2 câu cách 1 câu (t, t+2) -> cùng người nói (nhãn 1)
    diff_prob = same_speaker[order[:-1], order[1:]]
    same_prob = same_speaker[order[:-2], order[2:]]
    loss_speaker = (F.binary_cross_entropy(diff_prob, torch.zeros_like(diff_prob)) +
                    F.binary_cross_entropy(same_prob, torch.ones_like(same_prob))) / 2

    return loss_next + 0.3 * loss_speaker


# Kiểm tra nhanh trên 1 hội thoại
test_model = OrderModel().to(device)
item = train_data[0]
next_score, same_speaker = test_model(item["spk"], item["edge"], item["sem"])
loss = compute_loss(next_score, same_speaker, item["order"])
loss.backward()
print("next_score:", tuple(next_score.shape), "| same_speaker:", tuple(same_speaker.shape),
      "| loss ban đầu:", round(loss.item(), 4))
del test_model

next_score: (12, 12) | same_speaker: (12, 12) | loss ban đầu: 2.8253


In [16]:
# ===================== CELL 5: Tìm thứ tự tốt nhất + chấm điểm =====================
import itertools


def make_score_matrix(next_log_prob, same_speaker):
    # score[i, j] = 0.7 * log P(j nói ngay sau i) + 0.3 * log P(i và j KHÁC người nói)
    # -> thưởng cho bước đổi người nói, phạt nặng nếu 2 câu liền nhau cùng giọng
    return 0.7 * next_log_prob + 0.3 * np.log(np.clip(1 - same_speaker, 1e-6, 1))


def dp_order(score, first, second):
    """Bitmask DP: tìm đường đi qua mọi câu (bắt đầu first -> second) có tổng điểm lớn nhất.
    dp[mask][u] = điểm tốt nhất khi đã dùng các câu trong mask và đang đứng ở câu u."""
    N = len(score)
    NEG = -1e18
    dp = np.full((1 << N, N), NEG)
    parent = np.full((1 << N, N), -1, dtype=np.int64)
    bits = 1 << np.arange(N)                    # bits[v] = 1 << v

    start = (1 << first) | (1 << second)
    dp[start, second] = score[first, second]

    for mask in range(start, 1 << N):
        if (mask & start) != start:             # mask phải chứa sẵn 2 câu đầu
            continue
        row = dp[mask]
        if row.max() <= NEG / 2:                # trạng thái không tới được
            continue
        # cand[u, v] = điểm nếu đang ở u rồi nói tiếp câu v
        cand = row[:, None] + score
        best_u = cand.argmax(axis=0)            # với mỗi v: câu u tốt nhất đứng trước
        best_value = cand.max(axis=0)

        free = np.nonzero((mask & bits) == 0)[0]    # các câu chưa dùng
        new_masks = mask | bits[free]
        better = best_value[free] > dp[new_masks, free]
        dp[new_masks[better], free[better]] = best_value[free][better]
        parent[new_masks[better], free[better]] = best_u[free][better]

    # Lần ngược parent để lấy lại thứ tự
    mask = (1 << N) - 1
    order = [int(dp[mask].argmax())]
    while order[-1] != second:
        u = order[-1]
        prev = int(parent[mask, u])
        mask ^= 1 << u
        order.append(prev)
    order.append(first)
    return order[::-1]


def beam_order(score, first, second, width=64):
    """Beam Search: mỗi bước chỉ giữ 'width' đường đi tốt nhất (nhanh, gần đúng)."""
    N = len(score)
    beams = [(score[first, second], [first, second])]
    for _ in range(N - 2):
        candidates = []
        for total, path in beams:
            u = path[-1]
            for v in range(N):
                if v not in path:
                    candidates.append((total + score[u, v], path + [v]))
        candidates.sort(key=lambda c: c[0], reverse=True)
        beams = candidates[:width]
    return [int(x) for x in beams[0][1]]


def find_best_order(score, first, second, fast=False):
    N = len(score)
    if fast:                        # dùng khi chấm validation lúc train cho nhanh
        return beam_order(score, first, second, width=8)
    if N <= 18:                     # DP cho kết quả tối ưu tuyệt đối
        return dp_order(score, first, second)
    return beam_order(score, first, second, width=256)


def pair_score(pred_order, true_order):
    """Điểm của đề: 1 - (số cặp bị đảo thứ tự) / (tổng số cặp)."""
    n = len(true_order)
    if n < 2:
        return 1.0
    true_position = {chunk: pos for pos, chunk in enumerate(true_order)}
    positions = [true_position[chunk] for chunk in pred_order]
    wrong = 0
    for i in range(n):
        for j in range(i + 1, n):
            if positions[i] > positions[j]:
                wrong += 1
    return 1 - wrong / (n * (n - 1) / 2)


# ---- Tự kiểm tra: DP phải trùng với thử hết mọi hoán vị (N nhỏ) ----
for trial in range(20):
    N = 8
    score = np.random.randn(N, N)
    first, second = random.sample(range(N), 2)
    rest = [c for c in range(N) if c not in (first, second)]
    best = max(itertools.permutations(rest),
               key=lambda p: sum(score[a, b] for a, b in zip([first, second, *p], [second, *p])))
    brute = [first, second, *best]
    assert dp_order(score, first, second) == brute, "DP sai!"
print("DP khớp với vét cạn ✔")
print("pair_score đúng hết  :", pair_score([0, 1, 2, 3], [0, 1, 2, 3]))
print("pair_score ngược hết :", pair_score([3, 2, 1, 0], [0, 1, 2, 3]))

DP khớp với vét cạn ✔
pair_score đúng hết  : 1.0
pair_score ngược hết : 0.0


In [22]:
# ===================== CELL 6: Huấn luyện 5-Fold =====================
from tqdm.auto import tqdm   # thanh tiến trình

NUM_FOLDS = 5
EPOCHS = 15
LEARNING_RATE = 1e-3
s_history = []   # lưu trung bình S[i][j] sau mỗi epoch của mọi fold


@torch.no_grad()
def predict_order(models, item, fast=False):
    """Cho 1 hoặc nhiều mô hình dự đoán, lấy trung bình 2 ma trận rồi tìm thứ tự tốt nhất."""
    next_log_prob = 0
    same_speaker = 0
    for model in models:
        model.eval()
        next_score, same = model(item["spk"], item["edge"], item["sem"])
        next_log_prob = next_log_prob + F.log_softmax(next_score, dim=-1)
        same_speaker = same_speaker + same
    next_log_prob = (next_log_prob / len(models)).cpu().numpy()
    same_speaker = (same_speaker / len(models)).cpu().numpy()

    score = make_score_matrix(next_log_prob, same_speaker)
    return find_best_order(score, item["first"], item["second"], fast=fast)


def evaluate(model, data, fast=True):
    scores = [pair_score(predict_order([model], item, fast=fast), item["order"]) for item in data]
    return float(np.mean(scores))


@torch.no_grad()
def average_s(model, data):
    """Trung bình S[i][j] = P(câu j nói ngay sau câu i), đã softmax theo từng hàng.
    - s_true : trung bình trên các cặp liền nhau THẬT  (càng gần 1 càng tốt)
    - s_other: trung bình trên các cặp còn lại        (càng gần 0 càng tốt)"""
    model.eval()
    true_values, other_values = [], []
    for item in data:
        next_score, _ = model(item["spk"], item["edge"], item["sem"])
        S = F.softmax(next_score, dim=-1)                       # [N, N]
        N = S.shape[0]
        order = item["order"]
        is_true = torch.zeros(N, N, dtype=torch.bool, device=S.device)
        is_true[order[:-1], order[1:]] = True                   # ô (câu t, câu t+1)
        diagonal = torch.eye(N, dtype=torch.bool, device=S.device)
        is_other = ~is_true & ~diagonal
        true_values.append(S[is_true].mean().item())
        other_values.append(S[is_other].mean().item())
    return float(np.mean(true_values)), float(np.mean(other_values))


def train_one_fold(fold, train_part, val_part):
    model = OrderModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    save_path = f"/content/order_model_fold_{fold}.pt"
    best_score = -1

    epoch_bar = tqdm(range(1, EPOCHS + 1), desc=f"Fold {fold}", unit="epoch")
    for epoch in epoch_bar:
        # ---- Train: mỗi lần 1 hội thoại ----
        model.train()
        random.shuffle(train_part)
        total_loss = 0
        for item in tqdm(train_part, desc=f"  Epoch {epoch}", unit="hội thoại", leave=False):
            next_score, same_speaker = model(item["spk"], item["edge"], item["sem"])
            loss = compute_loss(next_score, same_speaker, item["order"])
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        # ---- Validation: điểm của đề + trung bình S[i][j] ----
        val_score = evaluate(model, val_part, fast=True)
        s_true, s_other = average_s(model, val_part)
        s_history.append({"fold": fold, "epoch": epoch, "s_true": s_true, "s_other": s_other})

        if val_score > best_score:
            best_score = val_score
            torch.save(model.state_dict(), save_path)

        epoch_bar.set_postfix(loss=f"{total_loss / len(train_part):.4f}",
                              val=f"{val_score:.4f}", best=f"{best_score:.4f}",
                              s_true=f"{s_true:.3f}", s_other=f"{s_other:.3f}")

    # Chấm lại mô hình tốt nhất bằng thuật toán chính xác (DP)
    model.load_state_dict(torch.load(save_path, map_location=device))
    exact_score = evaluate(model, val_part, fast=False)
    tqdm.write(f"==> Fold {fold}: val score (DP) = {exact_score:.4f}")
    tqdm.write("    Trung bình S[i][j] trên val sau mỗi epoch:")
    for h in s_history:
        if h["fold"] == fold:
            tqdm.write(f"    epoch {h['epoch']:02d} | s_true {h['s_true']:.4f} | s_other {h['s_other']:.4f}")
    tqdm.write("")
    return save_path, exact_score


kfold = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
fold_files = []
fold_scores = []
for fold, (train_idx, val_idx) in enumerate(kfold.split(train_data), start=1):
    train_part = [train_data[i] for i in train_idx]
    val_part = [train_data[i] for i in val_idx]
    path, score = train_one_fold(fold, train_part, val_part)
    fold_files.append(path)
    fold_scores.append(score)

print("Điểm từng fold:", [round(s, 4) for s in fold_scores])
print(f"Điểm trung bình 5 fold: {np.mean(fold_scores):.4f}  (thang 100: {100 * np.mean(fold_scores):.2f})")

Fold 1:   0%|          | 0/15 [00:00<?, ?epoch/s]

  Epoch 1:   0%|          | 0/1030 [00:00<?, ?hội thoại/s]

  Epoch 2:   0%|          | 0/1030 [00:00<?, ?hội thoại/s]

  Epoch 3:   0%|          | 0/1030 [00:00<?, ?hội thoại/s]

  Epoch 4:   0%|          | 0/1030 [00:00<?, ?hội thoại/s]

...

  Epoch 10:   0%|          | 0/1030 [00:00<?, ?hội thoại/s]

  Epoch 11:   0%|          | 0/1030 [00:00<?, ?hội thoại/s]

  Epoch 12:   0%|          | 0/1030 [00:00<?, ?hội thoại/s]

  Epoch 13:   0%|          | 0/1030 [00:00<?, ?hội thoại/s]

KeyboardInterrupt: 

In [21]:
# ===================== CELL 7: Dự đoán tập test và ghi answers.json =====================
# Nạp lại 5 mô hình tốt nhất của 5 fold
import glob
model_files = sorted(glob.glob("/content/order_model_fold_*.pt"))
if len(model_files) == 0:
    raise FileNotFoundError("Chưa có file order_model_fold_*.pt nào. Hãy chạy  6 trước.")
print("Dùng các mô hình:", model_files)

models = []
for path in model_files:
    model = OrderModel().to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    models.append(model)

# Dự đoán: trung bình 5 mô hình -> tìm thứ tự bằng DP -> đổi sang RANK để nộp
answers = {}
for item in test_data:
    order = predict_order(models, item, fast=False)
    answers[item["id"]] = order_to_rank(order)

with open("answers.json", "w") as f:
    json.dump(answers, f)
print(f"Đã ghi answers.json: {len(answers)} hội thoại")

# Kiểm tra mỗi đáp án là một hoán vị hợp lệ của 0..N-1
for item in test_data:
    assert sorted(answers[item["id"]]) == list(range(item["spk"].shape[0]))

# Tự chấm điểm nếu có đáp án tập test
if test_answers is not None:
    scores = [pair_score(rank_to_order(answers[cid]), rank_to_order(test_answers[cid])) for cid in answers]
    print(f"Điểm trên {TEST_SPLIT}: {100 * np.mean(scores):.2f} / 100")

Dùng các mô hình: ['/content/order_model_fold_1.pt']
Đã ghi answers.json: 100 hội thoại
Điểm trên test_leaderboard_a: 70.35 / 100
